# Stock Volatility Forecasting

## Project Overview

Forecasts **daily realized stock volatility** (%) over a 14-day horizon. Synthetic data spans 2 years with volatility clustering, regime shifts, and earnings-driven spikes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily realized volatility, predict the next 14 days. Volatility forecasts are essential for options pricing, portfolio risk management, and position sizing.

## Why This Project Matters

Volatility is the key input to options pricing (Black-Scholes), Value-at-Risk calculations, and dynamic portfolio allocation. Unlike price direction, volatility is more forecastable — it clusters in time. Accurate vol forecasts directly translate to better risk management and P&L.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily realized volatility
- Volatility clustering (GARCH-like behavior)
- Regime shifts between calm and turbulent periods
- Earnings-driven quarterly spikes
- Right-skewed distribution (typical for vol)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `volatility_pct` | Daily realized volatility (%, annualized proxy) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'volatility_pct'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# GARCH-like volatility clustering
vol = np.zeros(N_POINTS)
vol[0] = 15.0  # starting vol
omega, alpha, beta = 2.0, 0.15, 0.80

for i in range(1, N_POINTS):
    shock = np.random.normal(0, vol[i-1])
    vol[i] = np.sqrt(omega + alpha * shock**2 + beta * vol[i-1]**2)

# Regime shifts
for s in range(0, N_POINTS, 200):
    regime_start = min(s + np.random.randint(80, 160), N_POINTS - 40)
    dur = min(np.random.randint(20, 50), N_POINTS - regime_start)
    vol[regime_start:regime_start + dur] *= 1.5

# Quarterly earnings spikes
for q in range(0, N_POINTS, 90):
    earn_day = min(q + np.random.randint(85, 92), N_POINTS - 1)
    vol[earn_day] *= np.random.uniform(1.5, 2.5)
    if earn_day + 1 < N_POINTS:
        vol[earn_day + 1] *= 1.3

volatility_pct = np.maximum(vol, 3.0).round(2)

df = pd.DataFrame({'date': dates, 'volatility_pct': volatility_pct})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,volatility_pct
0,2022-01-01,15.00
1,2022-01-02,13.80
2,2022-01-03,12.44
3,2022-01-04,11.64
4,2022-01-05,12.56


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('volatility_pct Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("volatility_pct Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

volatility_pct Statistics:
count    730.000000
mean       6.378123
std        1.840189
min        3.550000
25%        5.090000
50%        6.160000
75%        7.300000
max       18.740000
Name: volatility_pct, dtype: float64

CV: 0.289
Skewness: 1.691


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        1.9 | RMSE:        1.9 | MAPE: 35.33%
Seasonal Naive (7)             | MAE:        3.6 | RMSE:        4.6 | MAPE: 66.96%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared  R-Squared      RMSE  Time Taken
Model                                                                        
GaussianProcessRegressor          230.875405 -16.682723  8.624635    0.058025
KernelRidge                       161.409303 -11.339177  7.204590    0.063179
QuantileRegressor                  55.510598  -3.193123  4.199861    0.074401
DummyRegressor                     52.486831  -2.960525  4.081714    0.005991
LassoLars                          39.954318  -1.996486  3.550359    0.008088
Lasso                              39.954318  -1.996486  3.550359    0.008001
ElasticNet                         26.378629  -0.952202  2.865686    0.009236
ExtraTreeRegressor                 24.373933  -0.797995  2.750175    0.008824
NuSVR                              23.621338  -0.740103  2.705538    0.031415
SVR                                22.832065  -0.679390  2.657920    0.038763


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        1.9 | RMSE:        2.4 | MAPE: 17.71%
FLAML Test (lgbm)              | MAE:        0.9 | RMSE:        1.6 | MAPE: 13.11%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        4.4 | RMSE:        4.5 | MAPE: 87.46%
SF AutoETS                     | MAE:        6.6 | RMSE:        6.7 | MAPE: 131.06%
SF SeasonalNaive               | MAE:        6.0 | RMSE:        6.2 | MAPE: 117.73%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model      MAE     RMSE       MAPE
 FLAML Test (lgbm) 0.865429 1.578151  13.112677
Naive (Last Value) 1.851429 1.942840  35.328117
      FLAML (lgbm) 1.940561 2.435076  17.713625
Seasonal Naive (7) 3.647857 4.622222  66.964002
      SF AutoARIMA 4.405813 4.531140  87.464921
  SF SeasonalNaive 5.991429 6.182170 117.734511
        SF AutoETS 6.570093 6.744984 131.063538

Top 3:
             model      MAE     RMSE      MAPE
 FLAML Test (lgbm) 0.865429 1.578151 13.112677
Naive (Last Value) 1.851429 1.942840 35.328117
      FLAML (lgbm) 1.940561 2.435076 17.713625


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 0.03, Std: 1.58


## Interpretation and Business Insight

- Volatility clusters strongly — high-vol days tend to follow high-vol days
- Regime shifts create sustained periods of elevated volatility
- Earnings announcements cause predictable quarterly spikes
- Volatility is mean-reverting — extreme levels eventually normalize
- The distribution is right-skewed (occasional very high values)

## Limitations

1. Synthetic GARCH — real vol depends on market microstructure, news flow
2. No VIX or implied volatility comparison
3. Single asset — portfolio vol needs cross-asset correlations
4. No intraday data (5-min realized vol is industry standard)
5. No options market data for implied vs realized comparison

## How to Improve This Project

1. Use GARCH/EGARCH for parametric volatility modeling
2. Add VIX as an implied volatility feature
3. Include intraday high-frequency data for realized vol calculation
4. Use HAR (Heterogeneous Autoregressive) model
5. Add news sentiment for event-driven vol prediction

## Production Considerations

- Real-time volatility estimation for options desks
- VaR and Expected Shortfall computation
- Dynamic position sizing and portfolio rebalancing
- Volatility surface calibration for derivatives pricing

## Common Mistakes

1. Using point forecasts when vol is inherently stochastic
2. Ignoring the leverage effect (vol rises more on down moves)
3. Not distinguishing realized vs implied volatility
4. Treating vol as stationary without checking for regime changes
5. Using daily data when intraday data is available

## Mini Challenge / Exercises

1. Fit a simple EWMA volatility model and compare
2. Detect volatility regimes using a rolling window approach
3. Build a vol-of-vol (volatility of volatility) measure
4. Compare forecast accuracy in calm vs turbulent regimes

## Final Summary / Key Takeaways

1. Volatility is more forecastable than price direction
2. Clustering is the dominant feature — persistence matters
3. GARCH-family models are the industry standard
4. Regime detection improves conditional forecasts
5. Real deployment needs intraday data and proper risk framework

In [18]:
import json
metrics = {
    'project': 'Stock Volatility Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Stock Volatility Forecasting — Complete ===")

Metrics saved.

=== Stock Volatility Forecasting — Complete ===
